## 第25章　权重初始化 —— 起点决定终点

> 本章目标：理解全零初始化的对称性陷阱（所有神经元更新完全相同），随机初始化的方差漂移（前向传播中激活值逐层爆炸或消失），以及 Xavier 和 Kaiming 初始化如何精确控制方差保持——让 100 层网络的激活值既不爆炸也不消失。
> 前置知识：第 15 章（正态/均匀分布）、第 17 章（方差）、第 21 章（归一化）



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
print("env ready")


### 25.1　全零初始化 —— 对称性灾难

如果把神经网络的所有权重初始化为 0，第一个悲剧立刻发生：每个神经元做完全相同的计算。前向传播时，同一层的每个神经元输出完全相同的值；反向传播时，每个神经元接收到完全相同的梯度；参数更新后——所有权重仍然完全相同。

无论训练多少轮，无论网络有多宽，同一层的所有神经元永远等价于一个神经元。**网络的有效宽度退化成了 1。** 这就是"对称性陷阱"。

📐 **定义　对称性问题（Symmetry Problem）**：全零（或全等）初始化导致同层神经元无法差异化——所有神经元学习到相同的特征，网络容量被完全浪费。

💻 **代码　全零初始化的对称性验证**


In [ ]:
import numpy as np

# 3 个样本, 4 个输入特征, 3 个隐藏神经元
np.random.seed(42)
X = np.random.randn(3, 4)
W = np.zeros((4, 3))  # 全零权重!
b = np.zeros(3)

# 前向传播
h = X @ W + b  # 所有神经元输出完全相同 (都是 0)
print(f"隐藏层输出:\n{h}")
print(f"所有列相等: {np.allclose(h[:, 0], h[:, 1]) and np.allclose(h[:, 0], h[:, 2])}")
print("→ 3 个神经元等效于 1 个——对称性扼杀了多样性")


> **关键洞察**：全零初始化是深度学习中最经典的"静默错误"——代码不会报错，loss 也会下降，但网络从未真正利用它的全部容量。只需把初始化换成随机值，对称性立刻打破。

---


### 25.2　随机初始化（小高斯）—— 方差漂移的隐患

用小的随机数（如 `np.random.randn * 0.01`）初始化解决了对称性。但引入了新问题：**前向传播中激活值的方差随层数漂移。**

如果权重太小，每层变换后信号方差缩小——100 层后激活值趋于 0，梯度也趋于 0（梯度消失）。如果权重太大，方差逐层放大——100 层后激活值爆炸到 Inf（梯度爆炸）。

📐 **问题核心**：随机初始化没有考虑**层与层之间的方差传递**。每经过一层 `h = X @ W`，输出的方差会乘以权重的方差——如果这个乘积 ≠ 1，方差就会指数级漂移。

---


### 25.3　Xavier 初始化 —— 保持前向/反向方差一致

Xavier（Glorot）初始化的核心思想：**让每层输出的方差等于输入的方差。** 对 `fan_in` 个输入、`fan_out` 个输出的线性层，从 `Uniform[-limit, limit]` 采样，其中：

$$limit = \sqrt{\frac{6}{fan\_in + fan\_out}}$$

这个公式推导自一个假设：激活函数是线性的（或 tanh 在零点附近的近似）。对于 tanh/sigmoid，Xavier 工作良好——但对 ReLU 不够。

📐 **定义　Xavier 初始化**：Uniform[-limit, limit]，limit = √(6/(fan_in+fan_out))。保持前向和反向传播的方差一致。适用于 tanh/sigmoid 激活函数。

---


### 25.4　Kaiming 初始化 —— 针对 ReLU 的改进

ReLU 把一半的输入（负值）清零了——这相当于"损失了一半的方差"。Kaiming（He）初始化对此做了补偿：

$$\text{std} = \sqrt{\frac{2}{fan\_in}}$$

比 Xavier 多一个 √2 因子。PyTorch 的 `nn.Linear` 默认就是 `kaiming_uniform_`。

📐 **定义　Kaiming 初始化**：N(0, √(2/fan_in)) 或 Uniform[-√(6/fan_in), √(6/fan_in)]。专为 ReLU 及其变体设计，补偿 ReLU 的"半数清零"效应。

---


### 25.5　代码动画：不同初始化下 100 层网络输出分布的变化

💻 **代码　四种初始化在 100 层 ReLU 网络中的方差演化**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n_layers = 100
hidden_dim = 256
batch_size = 512

init_methods = {
    "Kaiming": lambda din, dout: np.random.randn(din, dout) * np.sqrt(2.0 / din),
    "Xavier": lambda din, dout: np.random.uniform(
        -np.sqrt(6.0/(din+dout)), np.sqrt(6.0/(din+dout)), (din, dout)),
    "Small (0.01)": lambda din, dout: np.random.randn(din, dout) * 0.01,
    "Large (1.0)": lambda din, dout: np.random.randn(din, dout) * 1.0,
}

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

for ax, (name, init_fn) in zip(axes, init_methods.items()):
    x = np.random.randn(batch_size, hidden_dim)
    stds = [x.std()]
    for _ in range(n_layers):
        W = init_fn(hidden_dim, hidden_dim)
        x = np.maximum(0, x @ W)  # ReLU
        stds.append(x.std())
    ax.plot(range(n_layers + 1), stds, linewidth=1.5)
    ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5)
    ax.set_title(f'{name}\n(final std={stds[-1]:.4f})')
    ax.set_xlabel('Layer'); ax.set_ylabel('Activation Std')

plt.suptitle('100-Layer ReLU Network: Activation Std Evolution', fontsize=13)
plt.tight_layout(); plt.show()

print("Kaiming: std stays ~1 throughout — 100 layers, no drift")
print("Xavier: std gradually decays (did not compensate ReLU's 50% kill)")
print("Small:  signal vanishes within first few layers")
print("Large: signal explodes to infinity almost immediately")


> **关键洞察**：四张图一目了然——只有 Kaiming 保持激活值标准差在 1 附近 100 层不变。Xavier 缓慢衰减（√2 补偿不足），Small 瞬间归零，Large 瞬间爆炸。**初始化选错，你的网络在第一个 forward pass 就已经死了——无论优化器多强都救不回来。**

🔗 **AI 连接**：PyTorch 的 `nn.Linear` 默认 `kaiming_uniform_`，Transformer 的 `nn.MultiheadAttention` 内部也使用 Kaiming 风格的初始化。第 28 章手写两层网络时会显式调用 Kaiming 初始化权重。

---


**✏️ 习题**

> 在下方新建代码单元格作答。
1. （概念）全零初始化为什么不行？"对称性"具体指什么？
2. （概念）Kaiming 为什么比 Xavier 多一个 √2 因子？如果激活函数是 tanh 该用哪个？
3. （代码）在 50 层 ReLU 网络上对比 Xavier 和 Kaiming，画出每层输出的标准差曲线。验证 Kaiming 的标准差始终在 1 附近、Xavier 随层数衰减。
---
> 🔗 **章末钩子**：初始化 + 优化器都已就位。现在用它们完整训练一个模型——从生成数据到拟合直线，全流程闭环。
> 预览：下一章——**线性回归**，全流程闭环。
